# Live Strategy — Frozen Candidate C / CYCLE_ALWAYS_EXIT

**REAL MONEY NOTE:** only the explicitly armed smoke/full cells send orders. Run the read-only preflight first. Use V2, not V1.

Full validation: Q10 per market, 24h from the first complete M0 window, $50 start-to-current total-portfolio-value kill.
Smoke validation: Q1 per market for one complete synchronized strategy window, then automatic M5 flatten and stop.

In [ ]:
%cd "/Users/rchbeir/Desktop/Quant work/quant_proj"
!git checkout agent/mm-m1-m5-feasibility
!git pull --ff-only origin agent/mm-m1-m5-feasibility

In [ ]:
from importlib import reload
from quant_research.kalshi import mm_cycle_q10_live_strategy_v2 as LIVE
reload(LIVE)
print('Loaded:', LIVE.LIVE_VERSION)

## 1. Read-only preflight
This sends **no orders**. It checks auth, API limits, fee structure, clean account, starting equity, and that research recorders are stopped.

In [ ]:
LIVE.live_preflight(
    quote_size=10,
    runtime_hours=24,
    max_start_loss_usd=50,
    min_start_equity_usd=125,
    show=True,
)

## 2. Q1 one-window smoke test — REAL ORDERS
Q1 means **1 contract per eligible market**, not one contract total. It uses the exact frozen strategy for one synchronized M0–M5 window and then auto-flattens/stops.

In [ ]:
LIVE.start_live_smoke_q1_one_window(
    arm_phrase='Q1_ONE_WINDOW',
    max_start_loss_usd=50,
)

## 3. Status / log tail

In [ ]:
LIVE.live_status(show=True, tail_lines=30)

## 4. Emergency kill + flatten — REAL ORDERS
Use only if you want to stop immediately. It requests the background process to trigger its order group, cancel, and reduce-only IOC flatten. If the process does not respond, the notebook runs a direct cleanup fallback.

In [ ]:
# LIVE.kill_and_flatten_live(arm_phrase='KILL_AND_FLATTEN')

## 5. Full Q10 24-hour validation — REAL ORDERS
Run this only after the frozen OOS audit and after the Q1 smoke run finishes flat.

In [ ]:
LIVE.start_live_cycle_q10(
    arm_phrase='LIVE_Q10_24H',
    runtime_hours=24,
    max_start_loss_usd=50,
    min_start_equity_usd=125,
)

## 6. Final status
At automatic completion, `final_summary.json` reports account PnL, max peak drawdown, flat verification, final positions/resting orders and shutdown reason. Raw market + execution logs remain in the session folder for the formal live-vs-shadow analysis.

In [ ]:
LIVE.live_status(show=True, tail_lines=50)